In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [3]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


# Milestone 1


### ****1. Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?**** 

In [4]:
print(f"Total answer distribution {train_df['answer'].value_counts()}\n")
print('----------------------------------------------------------------\n')

print(f"Most frequent : {train_df['answer'].value_counts().max()}")

print(f"least frequent : {train_df['answer'].value_counts().min()}")

print(f"Sum of most frequent and least frequent : {train_df['answer'].value_counts().max() + train_df['answer'].value_counts().min()}" )



Total answer distribution answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

----------------------------------------------------------------

Most frequent : 490
least frequent : 324
Sum of most frequent and least frequent : 814


### ****2 . After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv**** 

In [5]:
train_df['prompt'].str.len()

0       142
1        43
2       182
3       129
4       110
       ... 
1995     72
1996    104
1997     93
1998    266
1999    140
Name: prompt, Length: 2000, dtype: int64

In [6]:
train_df['prompt']

0       Pick the best possible answer: What is Martin ...
1             What is accelerator-based light-ion fusion?
2       Determine the correct option: What is the term...
3       Select the most accurate option: What is Marti...
4       Identify the correct statement: What is the co...
                              ...                        
1995    What is the piezoelectric strain coefficient f...
1996    Identify the correct statement: What is the sy...
1997    Determine the correct option: What does Earnsh...
1998    Identify the correct statement: What is the re...
1999    Pick the best possible answer: Which of the fo...
Name: prompt, Length: 2000, dtype: object

In [7]:
import string

train_df['cleaned_prompt'] = train_df['prompt'].apply(
    lambda text: ''.join(
        ch.lower() for ch in text
        if ch not in string.punctuation
    ).split()
)


In [8]:
unique_words= set()

for words in train_df['cleaned_prompt']:
    unique_words.update(words)

print(f'No of unique words in prompt : {len(unique_words)}')

No of unique words in prompt : 859


### ****3. Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?****

In [9]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [10]:

print(len(ENGLISH_STOP_WORDS))

318


In [11]:
train_df['cleaned_prompt']

0       [pick, the, best, possible, answer, what, is, ...
1          [what, is, acceleratorbased, lightion, fusion]
2       [determine, the, correct, option, what, is, th...
3       [select, the, most, accurate, option, what, is...
4       [identify, the, correct, statement, what, is, ...
                              ...                        
1995    [what, is, the, piezoelectric, strain, coeffic...
1996    [identify, the, correct, statement, what, is, ...
1997    [determine, the, correct, option, what, does, ...
1998    [identify, the, correct, statement, what, is, ...
1999    [pick, the, best, possible, answer, which, of,...
Name: cleaned_prompt, Length: 2000, dtype: object

In [12]:
train_df['non_stop_words'] = train_df['cleaned_prompt'].apply(lambda words: [word for word in words
                                                                        if word not in ENGLISH_STOP_WORDS])

In [13]:
print(f'words left in the prompt for Row ID 1 after filtering are: {train_df['non_stop_words'][0]}\n')

print(f'No of words left in the prompt for Row ID 1 after filtering is: {len(train_df['non_stop_words'][0])}')

words left in the prompt for Row ID 1 after filtering are: ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']

No of words left in the prompt for Row ID 1 after filtering is: 13


### ****4. Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?****

In [14]:
train_df.head()

,id,prompt,A,B,C,D,E,answer,cleaned_prompt,non_stop_words
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,"[pick, the, best, possible, answer, what, is, ...","[pick, best, possible, answer, martin, heidegg..."
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,"[what, is, acceleratorbased, lightion, fusion]","[acceleratorbased, lightion, fusion]"
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,"[determine, the, correct, option, what, is, th...","[determine, correct, option, term, used, astro..."
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,"[select, the, most, accurate, option, what, is...","[select, accurate, option, martin, heideggers,..."
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,"[identify, the, correct, statement, what, is, ...","[identify, correct, statement, concept, simult..."


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer



In [16]:
combined_text = (
    train_df['prompt'].fillna('') + ' ' +
    train_df['A'].fillna('') + ' ' +
    train_df['B'].fillna('') + ' ' +
    train_df['C'].fillna('') + ' ' +
    train_df['D'].fillna('') + ' ' +
    train_df['E'].fillna(''))

In [17]:
vectorizer = TfidfVectorizer(stop_words='english')

X = vectorizer.fit_transform(combined_text)


In [18]:
print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Shape:", X.shape)

Vocabulary size: 2762
Shape: (2000, 2762)


### ****5. Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).****

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text)

prompt = train_df.loc[0,'prompt']
a = train_df.loc[0,'A']


prompt_vector = vectorizer.transform([prompt])
a_vector = vectorizer.transform([a])


 

In [20]:
similarity = cosine_similarity(prompt_vector, a_vector)[0, 0]
print(f'Cosine similarity: {similarity}')

Cosine similarity: 0.27202429519891635


### ***6. Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.***

In [38]:
correct = 0

for _, row in train_df.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    scores = {}

    for opt in ["A","B","C","D","E"]:
        option_vec = vectorizer.transform([row[opt]])

        scores[opt] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0,0]

    predicted = max(scores, key=scores.get)

    if predicted == row["answer"]:
        correct += 1

percentage = correct / len(train_df) * 100

print(f'Percentage of instances : {percentage}')


Percentage of instances : 13.55


### ***7. If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?***

In [28]:
ranked

[('C', np.float64(0.5876662698435949)),
 ('D', np.float64(0.5385329310060211)),
 ('B', np.float64(0.3036286931760937)),
 ('A', np.float64(0.27202429519891635)),
 ('E', np.float64(0.23662770351943294))]

In [29]:
score = 0
answer = 'C'

top3= ['C','A','B']

if answer in top3:
    score = 1 / (top3.index(answer) + 1)

print(f'score is: {score}')

score is: 1.0


### ***8. If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?***

In [30]:
top= ['D','B','E']
ans='B'

if ans in top:
    score = 1 / (top.index(ans) + 1)


print(f'score is: {score}')

score is: 0.5


### ***9. The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?***

In [31]:
print(f"Most frequent : {train_df['answer'].value_counts()}" )


Most frequent : answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


In [32]:
freq_order =['B','C','A']
score = 0
for _,row in train_df.iterrows():
    answer = row['answer']

    if answer in freq_order:
        score += 1/(freq_order.index(answer)+1)

map3 = score/len(train_df)
print(f'MAP@3: {map3}')
    

MAP@3: 0.4212500000000017


### ***10. The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?***

In [49]:
points=[]
for _,row in train_df.iterrows():
    prompt_vec = vectorizer.transform([row['prompt']])

    scores={}

    for opt in ['A','B','C','D','E']:
        opt_vec = vectorizer.transform([row[opt]])
        scores[opt]=cosine_similarity(prompt_vec,opt_vec )[0][0]

    top3 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]
    top3_options = [option for option, score in top3]
        
    answer = row['answer']
        
    if answer in top3_options:
        
        points.append(1/(top3_options.index(row['answer'])+1))


map3_score = sum(points)/len(train_df)

print(f' MAP@3 score: {map3_score}')
        

    
    

 MAP@3 score: 0.2961666666666667
